## <span style="color:blue;">**Explore the Data: Summarized Findings**</span>
### 1. **Data Quality Issues**
#### * **Transaction Matching:** Difficult to analyze without matching user and product info to transactions.
#### * **Missing Barcodes:** Barcodes are missing in both Products and Transaction tables. This field is crucial for joining these tables.
#### * **Duplicate Barcodes:** Products table contains duplicate barcodes. Barcodes should be unique as they are the primary key for joining to the Transaction table.
#### * **Receipt ID Duplication:** Transaction table has two records for each Receipt ID. Likely, only records with quantity and sale numbers should be kept.
#### * **Suspicious Birth Dates:** Birth dates go back to the early 1900s, which seems unlikely.
#### * **Duplicate Records:** Duplicate records exist in both Products and Transaction tables.
#### * **Incorrect Data Types:** Dates are not in datetime format, and Quantity and Sales are not numeric.<br>
### 2. **Fields Challenging to Understand**
#### * **Final_Qty and Final_Sale:** Duplicate rows for Receipt ID with differing Final_Qty and Final_Sale are challenging. It appears only the record with non-null numeric values for both fields should be kept.
#### * **User IDs and Barcodes:** The low match rates for user IDs and barcodes to the transaction table are challenging to understand and require further investigation.

In [1]:
# Import modules
import pandas as pd
import numpy as np

In [2]:
# Read files into dataframes
df_user = pd.read_csv('USER_TAKEHOME.csv')
df_products = pd.read_csv('PRODUCTS_TAKEHOME.csv')
df_transaction = pd.read_csv('TRANSACTION_TAKEHOME.csv')

In [3]:
# Set display options to prevent wrapping
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Print the sample rows of each DataFrame
print("User Data:")
print(df_user.head())

print("\nProducts Data:")
print(df_products.head())

print("\nTransaction Data:")
print(df_transaction.head())


User Data:
                         ID               CREATED_DATE                 BIRTH_DATE STATE LANGUAGE  GENDER
0  5ef3b4f17053ab141787697d  2020-06-24 20:17:54.000 Z  2000-08-11 00:00:00.000 Z    CA   es-419  female
1  5ff220d383fcfc12622b96bc  2021-01-03 19:53:55.000 Z  2001-09-24 04:00:00.000 Z    PA       en  female
2  6477950aa55bb77a0e27ee10  2023-05-31 18:42:18.000 Z  1994-10-28 00:00:00.000 Z    FL   es-419  female
3  658a306e99b40f103b63ccf8  2023-12-26 01:46:22.000 Z                        NaN    NC       en     NaN
4  653cf5d6a225ea102b7ecdc2  2023-10-28 11:51:50.000 Z  1972-03-19 00:00:00.000 Z    PA       en  female

Products Data:
          CATEGORY_1              CATEGORY_2                   CATEGORY_3 CATEGORY_4                                       MANUFACTURER            BRAND       BARCODE
0  Health & Wellness           Sexual Health  Conductivity Gels & Lotions        NaN                                                NaN              NaN  7.964944e+11
1        

In [4]:
# Check for missing values in each column
# Replace empty strings and whitespace with NaN
df_user.replace(r'^\s*$', np.nan, regex=True, inplace=True)
df_products.replace(r'^\s*$', np.nan, regex=True, inplace=True)
df_transaction.replace(r'^\s*$', np.nan, regex=True, inplace=True)

print("User Data: Missing Values")
print(df_user.isnull().sum())

print("\nProducts Data: Missing Values")
print(df_products.isnull().sum())

print("\nTransaction Data: Missing Values")
print(df_transaction.isnull().sum())


User Data: Missing Values
ID                  0
CREATED_DATE        0
BIRTH_DATE       3675
STATE            4812
LANGUAGE        30508
GENDER           5892
dtype: int64

Products Data: Missing Values
CATEGORY_1         111
CATEGORY_2        1424
CATEGORY_3       60566
CATEGORY_4      778093
MANUFACTURER    226474
BRAND           226472
BARCODE           4025
dtype: int64

Transaction Data: Missing Values
RECEIPT_ID            0
PURCHASE_DATE         0
SCAN_DATE             0
STORE_NAME            0
USER_ID               0
BARCODE            5762
FINAL_QUANTITY        0
FINAL_SALE        12500
dtype: int64


#### - Barcode has missing values in the Products table. This field has a 1-to-many relationship with the Transactions table.
#### - Barcode also has missing values in the Transactions table so there will be no matches to products for 5762 records.

In [5]:
# Check the number of unique values in each column and total rows in each table
print("User Data: Unique Values")
print(df_user.nunique())
print(f"Total rows: {df_user.shape[0]}")

print("\nProducts Data: Unique Values")
print(df_products.nunique())
print(f"Total rows: {df_products.shape[0]}")

print("\nTransaction Data: Unique Values")
print(df_transaction.nunique())
print(f"Total rows: {df_transaction.shape[0]}")


User Data: Unique Values
ID              100000
CREATED_DATE     99942
BIRTH_DATE       54721
STATE               52
LANGUAGE             2
GENDER              11
dtype: int64
Total rows: 100000

Products Data: Unique Values
CATEGORY_1          27
CATEGORY_2         121
CATEGORY_3         344
CATEGORY_4         127
MANUFACTURER      4354
BRAND             8122
BARCODE         841342
dtype: int64
Total rows: 845552

Transaction Data: Unique Values
RECEIPT_ID        24440
PURCHASE_DATE        89
SCAN_DATE         24440
STORE_NAME          954
USER_ID           17694
BARCODE           11027
FINAL_QUANTITY       87
FINAL_SALE         1434
dtype: int64
Total rows: 50000


#### - There are 100,000 unique users in User table.
#### - There are 841,342 unique product bardcode with some duplication in barcode in Products table as total row count is higher.
#### - There are 50,000 transaction in Transaction table with approximately half have duplicate receipt ID.

In [6]:
# Count duplicates in the 'Receipt ID' column of the products table
duplicate_receiptid = df_transaction[df_transaction.duplicated(subset=['RECEIPT_ID'])]
count_duplicate_receiptid = duplicate_receiptid['RECEIPT_ID'].nunique()
print(f"Number of duplicate Receipt_ID in Transaction table: {count_duplicate_receiptid}")

# Sort the DataFrame by the 'Receipt ID' column
df_transaction_sorted = df_transaction.sort_values(by='RECEIPT_ID')

# Print a sample of the duplicate 'Receipt ID'
print("\nTransaction Data: Duplicate 'Receipt ID'")
print(df_transaction_sorted.head(10))

Number of duplicate Receipt_ID in Transaction table: 24440

Transaction Data: Duplicate 'Receipt ID'
                                 RECEIPT_ID PURCHASE_DATE                  SCAN_DATE STORE_NAME                   USER_ID       BARCODE FINAL_QUANTITY FINAL_SALE
0      0000d256-4041-4a3e-adc4-5623fb6e0c99    2024-08-21  2024-08-21 14:19:06.539 Z    WALMART  63b73a7f3d310dceeabd4758  1.530001e+10           1.00        NaN
41567  0000d256-4041-4a3e-adc4-5623fb6e0c99    2024-08-21  2024-08-21 14:19:06.539 Z    WALMART  63b73a7f3d310dceeabd4758  1.530001e+10           1.00       1.54
1      0001455d-7a92-4a7b-a1d2-c747af1c8fd3    2024-07-20  2024-07-20 09:50:24.206 Z       ALDI  62c08877baa38d1a1f6c211a           NaN           zero       1.49
39291  0001455d-7a92-4a7b-a1d2-c747af1c8fd3    2024-07-20  2024-07-20 09:50:24.206 Z       ALDI  62c08877baa38d1a1f6c211a           NaN           1.00       1.49
2      00017e0a-7851-42fb-bfab-0baa96e23586    2024-08-18  2024-08-19 15:38:56.813 Z    W

#### - Duplicate records for Receipt ID needs clarification to better understand the reason why. 
#### - Suspect that records with both values for Final Quantity and Final Sale should be retained for analysis.
#### - Final Quantity is numeric but contains text (i.e. "zero").

In [7]:
# Check for matches in transaction table to user and products 
# Transaction and User tables using user id
merged_df1 = pd.merge(df_transaction, df_user, left_on='USER_ID', right_on='ID', how='left', indicator=True)
matches_df1 = merged_df1[merged_df1['_merge'] == 'both'].shape[0]

# Transaction and Products tables using barcode
# Drop null barcodes in transaction
df_transaction_unique = df_transaction.dropna(subset=['BARCODE'])
merged_df2 = pd.merge(df_transaction_unique, df_products, left_on='BARCODE', right_on='BARCODE', how='left', indicator=True)
matches_df2 = merged_df2[merged_df2['_merge'] == 'both'].shape[0]

# Calculate total records that are not null in Transactions table
total_transactions_userid_notnull = df_transaction.dropna(subset=['USER_ID']).shape[0]
total_transactions_barcode_notnull = df_transaction.dropna(subset=['BARCODE']).shape[0]

# Calculate duplicate records for primary keys
duplicate_user_id = df_user['ID'].duplicated().sum()
print(f"Number of duplicate user id in user table: {duplicate_user_id}")
duplicate_barcode = df_products['BARCODE'].duplicated().sum()
print(f"Number of duplicate barcode in products table: {duplicate_barcode}\n")

print(f'Total records in Transaction table where user id is not null: {total_transactions_userid_notnull}')
print(f'Total records in Transaction table where barcode is not null: {total_transactions_barcode_notnull}\n')

print(f'Matches in Transaction table to User table using user id: {matches_df1}')
print(f'Matches in Transaction table to Products table using user barcode: {matches_df2}\n')

# Calculate the percentage of matches and print the results
percentage_matches_user = (matches_df1 / total_transactions_userid_notnull) * 100
percentage_matches_products = (matches_df2 / total_transactions_barcode_notnull) * 100

print(f'Percentage of matches in Transaction table to User table using user id: {percentage_matches_user:.2f}%')
print(f'Percentage of matches in Transaction table to Products table using barcode: {percentage_matches_products:.2f}%')




Number of duplicate user id in user table: 0
Number of duplicate barcode in products table: 4209

Total records in Transaction table where user id is not null: 50000
Total records in Transaction table where barcode is not null: 44238

Matches in Transaction table to User table using user id: 262
Matches in Transaction table to Products table using user barcode: 24854

Percentage of matches in Transaction table to User table using user id: 0.52%
Percentage of matches in Transaction table to Products table using barcode: 56.18%


#### - Cannot match user to Transactions. Less than 1%.
#### - About half products can match to Transactions.
#### - Extremely difficult to conduct meaningful analysis when user and product information cannot be matched to transactions.

In [8]:
df_user['BIRTH_DATE'] = pd.to_datetime(df_user['BIRTH_DATE'], errors='coerce')
birth_date_counts = df_user['BIRTH_DATE'].value_counts().sort_index()
print("Birth Date counts:")
print(birth_date_counts)

df_user['CREATED_DATE'] = pd.to_datetime(df_user['CREATED_DATE'], errors='coerce')
create_date_counts = df_user['CREATED_DATE'].value_counts().sort_index()
print("\nCreate Date counts:")
print(create_date_counts)



Birth Date counts:
BIRTH_DATE
1900-01-01 00:00:00+00:00    2
1900-11-01 08:00:00+00:00    1
1900-12-08 00:00:00+00:00    1
1901-01-01 05:00:00+00:00    4
1901-01-01 06:00:00+00:00    1
                            ..
2021-03-06 06:00:00+00:00    1
2021-03-17 05:00:00+00:00    1
2021-12-23 08:00:00+00:00    1
2022-03-01 05:00:00+00:00    1
2022-04-03 07:00:00+00:00    1
Name: count, Length: 54721, dtype: int64

Create Date counts:
CREATED_DATE
2014-04-18 23:14:55+00:00    1
2014-04-30 22:27:27+00:00    1
2014-05-02 21:09:30+00:00    1
2014-05-02 21:10:32+00:00    1
2014-05-02 21:11:42+00:00    1
                            ..
2024-09-11 17:58:12+00:00    1
2024-09-11 17:58:29+00:00    1
2024-09-11 17:58:57+00:00    1
2024-09-11 17:59:01+00:00    1
2024-09-11 17:59:15+00:00    1
Name: count, Length: 99942, dtype: int64


#### - Suspicious dates for birth date as there are dates in early 1900.

In [9]:
# Check for duplicate rows
print("User Data: Duplicate Rows")
print(df_user.duplicated().sum())

print("\nProducts Data: Duplicate Rows")
print(df_products.duplicated().sum())

print("\nTransaction Data: Duplicate Rows")
print(df_transaction.duplicated().sum())



User Data: Duplicate Rows
0

Products Data: Duplicate Rows
215

Transaction Data: Duplicate Rows
171


#### - Duplicate records in the Products and Transaction tables.

In [10]:
# Count duplicates in the 'Barcode' column of the products table
duplicate_product_barcodes = df_products[df_products.duplicated(subset=['BARCODE'])]
count_duplicate_product_barcodes = duplicate_product_barcodes['BARCODE'].nunique()
print(f"Number of duplicate Barcodes in products table: {count_duplicate_product_barcodes}")

# Sort the DataFrame by the 'Barcode' column
df_products_sorted = df_products.sort_values(by='BARCODE')

# Filter out rows with duplicate barcodes
duplicate_product_barcodes = df_products_sorted[df_products_sorted.duplicated(subset=['BARCODE'], keep=False)]

# Print a sample of the duplicate rows
print("\nProducts Data: Duplicate Rows sorted by 'Barcode'")
print(duplicate_product_barcodes.head(6))


Number of duplicate Barcodes in products table: 185

Products Data: Duplicate Rows sorted by 'Barcode'
       CATEGORY_1 CATEGORY_2        CATEGORY_3 CATEGORY_4              MANUFACTURER            BRAND   BARCODE
349945     Snacks      Candy  Confection Candy        NaN              MARS WRIGLEY        STARBURST  400510.0
99568      Snacks      Candy  Confection Candy        NaN              MARS WRIGLEY        STARBURST  400510.0
841230     Snacks      Candy   Chocolate Candy        NaN              MARS WRIGLEY            M&M'S  404310.0
139121     Snacks      Candy   Chocolate Candy        NaN  PLACEHOLDER MANUFACTURER  BRAND NOT KNOWN  404310.0
684662     Snacks   Crackers   Graham Crackers        NaN              TRADER JOE'S     TRADER JOE'S  438711.0
274321     Snacks   Crackers   Graham Crackers        NaN              TRADER JOE'S     TRADER JOE'S  438711.0


In [11]:
# Check the data types of each column
print("User Data: Data Types")
print(df_user.dtypes)

print("\nProducts Data: Data Types")
print(df_products.dtypes)

print("\nTransaction Data: Data Types")
print(df_transaction.dtypes)


User Data: Data Types
ID                           object
CREATED_DATE    datetime64[ns, UTC]
BIRTH_DATE      datetime64[ns, UTC]
STATE                        object
LANGUAGE                     object
GENDER                       object
dtype: object

Products Data: Data Types
CATEGORY_1       object
CATEGORY_2       object
CATEGORY_3       object
CATEGORY_4       object
MANUFACTURER     object
BRAND            object
BARCODE         float64
dtype: object

Transaction Data: Data Types
RECEIPT_ID         object
PURCHASE_DATE      object
SCAN_DATE          object
STORE_NAME         object
USER_ID            object
BARCODE           float64
FINAL_QUANTITY     object
FINAL_SALE         object
dtype: object


#### - Dates are not Datetime data types.
#### - Quantity and Sale are not numeric. 